In [1]:
# Import Required Libraries
from pathlib import Path
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import cv2
import random
import numpy as np
import pandas as pd
import dlib

In [2]:
import os
import cv2
from pathlib import Path
import dlib
import numpy as np

# Path ke predictor
predictor_path = 'shape_predictor_81_face_landmarks.dat'

# Inisialisasi detektor dan predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(predictor_path)

def calculate_distance(point1, point2):
    """Menghitung jarak Euclidean antara dua titik."""
    return np.linalg.norm(np.array(point1) - np.array(point2))

def is_frontal_face(img, threshold=30.0):

    # Pastikan gambar dalam format grayscale
    if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Deteksi wajah
    dets = detector(img, 0)
    for d in dets:
        # Deteksi landmark wajah
        shape = predictor(img, d)
        landmarks = np.array([[p.x, p.y] for p in shape.parts()])

        # Landmark untuk hidung, pipi kiri, dan pipi kanan
        try:
            nose = landmarks[30]
            left_cheek = landmarks[2]
            right_cheek = landmarks[14]

            # Hitung jarak dari hidung ke kedua sisi wajah
            left_distance = calculate_distance(nose, left_cheek)
            right_distance = calculate_distance(nose, right_cheek)

            # Periksa apakah wajah menghadap ke depan
            return abs(left_distance - right_distance) <= threshold
        except IndexError:
            # Jika landmark tidak mencukupi, kembalikan False
            return False
    return False

def clean_non_frontal_faces(img_dir):
 
    p = Path(img_dir)
    dirs = p.glob('*')

    for dir in dirs:
        for file in dir.glob('*.jpg'):
            try:
                img = cv2.imread(str(file))

                if img is not None:
                    # Hapus gambar jika wajah tidak menghadap ke depan
                    if not is_frontal_face(img):
                        os.remove(file)
                        print(f"Removed non-frontal face: {file}")
            except OSError:
                print(f"Skipped corrupted image: {file}")

# Path direktori dataset
train_dir = "FaceShape2/training_set"
test_dir = "FaceShape2/testing_set"

# Bersihkan gambar yang tidak menghadap ke depan dari training dan test set
clean_non_frontal_faces(train_dir)
clean_non_frontal_faces(test_dir)


Removed non-frontal face: FaceShape2\training_set\Heart\heart (10).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (1000).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (109).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (110).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (111).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (116).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (118).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (120).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (121).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (128).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (134).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (143).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (153).jpg
Removed non-frontal face: FaceShape2\training_set\Heart\heart (1

In [3]:
import os
import random
from pathlib import Path
import shutil

def balance_data(img_dir):
    """
    Menyeimbangkan jumlah data di setiap label dengan undersampling.
    """
    p = Path(img_dir)
    label_dirs = list(p.glob('*'))
    
    # Hitung jumlah data per label
    label_counts = {label.name: len(list(label.glob('*.jpg'))) for label in label_dirs}
    print("Jumlah data awal per label:", label_counts)
    
    # Tentukan jumlah minimum data per label
    min_count = min(label_counts.values())
    print(f"Jumlah data per label setelah penyeimbangan: {min_count}")
    
    # Undersampling untuk menyeimbangkan data
    for label_dir in label_dirs:
        images = list(label_dir.glob('*.jpg'))
        if len(images) > min_count:
            # Hapus gambar secara acak untuk menyisakan `min_count` gambar
            images_to_remove = random.sample(images, len(images) - min_count)
            for img in images_to_remove:
                os.remove(img)
                print(f"Removed: {img}")

    # Verifikasi hasil akhir
    label_counts_after = {label.name: len(list(label.glob('*.jpg'))) for label in label_dirs}
    print("Jumlah data setelah penyeimbangan:", label_counts_after)

# Path direktori dataset
train_dir = "FaceShape2/training_set"
test_dir = "FaceShape2/testing_set"

# Menyeimbangkan data
balance_data(train_dir)
balance_data(test_dir)


Jumlah data awal per label: {'Heart': 525, 'Oblong': 471, 'Oval': 511, 'Round': 505, 'Square': 549}
Jumlah data per label setelah penyeimbangan: 471
Removed: FaceShape2\training_set\Heart\heart (470).jpg
Removed: FaceShape2\training_set\Heart\heart (863).jpg
Removed: FaceShape2\training_set\Heart\heart (682).jpg
Removed: FaceShape2\training_set\Heart\heart (642).jpg
Removed: FaceShape2\training_set\Heart\heart (327).jpg
Removed: FaceShape2\training_set\Heart\heart (107).jpg
Removed: FaceShape2\training_set\Heart\heart (342).jpg
Removed: FaceShape2\training_set\Heart\heart (309).jpg
Removed: FaceShape2\training_set\Heart\heart (80).jpg
Removed: FaceShape2\training_set\Heart\heart (146).jpg
Removed: FaceShape2\training_set\Heart\heart (279).jpg
Removed: FaceShape2\training_set\Heart\heart (274).jpg
Removed: FaceShape2\training_set\Heart\heart (997).jpg
Removed: FaceShape2\training_set\Heart\heart (53).jpg
Removed: FaceShape2\training_set\Heart\heart (307).jpg
Removed: FaceShape2\training

In [2]:
import cv2
import numpy as np
from pathlib import Path
import random

def load_dataset(img_dir):
    p = Path(img_dir)
    dirs = p.glob('*')
    img_list = []

    for dir in dirs:
        label = dir.name  # Label dari nama folder
        for file in dir.glob('*.jpg'):
            try:
                img = cv2.imread(str(file))
                if img is not None:
                    img_list.append((img, label))
            except OSError:
                print(f"Skipped corrupted image: {file}")
    return img_list

# Path dataset
train_dir = "FaceShape2/training_set"
test_dir = "FaceShape2/testing_set"

# Load dataset
train_img = load_dataset(train_dir)
test_img = load_dataset(test_dir)

# Menampilkan panjang dataset (jumlah gambar)
print(f"Jumlah gambar dalam train_img: {len(train_img)}")
# Menampilkan panjang dataset (jumlah gambar)
print(f"Jumlah gambar dalam test_img: {len(test_img)}")


Jumlah gambar dalam train_img: 2354
Jumlah gambar dalam test_img: 560


In [3]:
def detectFace(image):
    """Mendeteksi wajah menggunakan Haar Cascade dan mengembalikan bounding box wajah terbesar."""
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    if len(faces) > 0:
        largest_face = max(faces, key=lambda face: face[2] * face[3])  # [x, y, w, h]
        return largest_face
    else:
        return None


def cropFace(image, face, target_size=(224, 224)):
    """Mencrop wajah dari gambar dengan menambahkan buffer dan resize dengan menjaga aspect ratio."""
    if face is not None:
        x, y, w, h = face
        buffer_top = int(h * 0.2)
        buffer_bottom = int(h * 0.2)
        buffer_left = int(w * 0.1)
        buffer_right = int(w * 0.1)
        y_top = max(0, y - buffer_top)
        y_bottom = min(image.shape[0], y + h + buffer_bottom)
        x_left = max(0, x - buffer_left)
        x_right = min(image.shape[1], x + w + buffer_right)
        cropped_face = image[y_top:y_bottom, x_left:x_right]
        old_h, old_w = cropped_face.shape[:2]
        target_w, target_h = target_size
        scale = min(target_w / old_w, target_h / old_h)
        new_w = int(old_w * scale)
        new_h = int(old_h * scale)
        resized_face = cv2.resize(cropped_face, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((target_h, target_w, 3), dtype=np.uint8)
        x_offset = (target_w - new_w) // 2
        y_offset = (target_h - new_h) // 2
        canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized_face
        return canvas
    else:
        return None


def preprocessing(image):
    face = detectFace(image)
    if face is not None:
        cropped_face = cropFace(image, face)
        return cropped_face.astype(np.uint8)
    else:
        return None


def standarized_input(image):

    image = preprocessing(image)
    if image is not None:
        
        return image.astype(np.uint8)
    else:
        return None


def preprocess(img_list):
    std_img_list = []
    std_label_list = []

    for item in img_list:
        image = item[0]
        label = item[1]

        # Preprocessing dan standarisasi gambar
        std_img = standarized_input(image)
        if std_img is not None:
            std_img_list.append(std_img)
            std_label_list.append(label)

    return std_img_list, std_label_list


# Memanggil preprocess pada dataset training dan testing
train_std_img_list, train_std_label_list = preprocess(train_img)
test_std_img_list, test_std_label_list = preprocess(test_img)


In [4]:
# Path ke predictor
predictor_path = 'shape_predictor_81_face_landmarks.dat'

# Inisialisasi detektor dan predictor
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor(predictor_path)

def detect_landmarks(std_img_list):
    detected_images = []
    for img in std_img_list:
        # Deteksi wajah pada gambar
        dets = detector(img, 0)
        for k, d in enumerate(dets):
            # Dapatkan landmark wajah
            shape = predictor(img, d)
            landmarks = np.matrix([[p.x, p.y] for p in shape.parts()])
        
        # Simpan gambar yang sudah diidentifikasi landmark ke dalam list
        detected_images.append(landmarks)
    
    return detected_images

# Melakukan deteksi pada train_std_img_list dan test_std_img_list
train_landmark_images = detect_landmarks(train_std_img_list)
test_landmark_images = detect_landmarks(test_std_img_list)

In [9]:
# Fungsi untuk menghitung panjang garis
def line_length(point1, point2):
    return np.sqrt((point2[0] - point1[0]) ** 2 + (point2[1] - point1[1]) ** 2)

def hitungsudut(p1, p2, p3):
    # Buat vektor dari titik-titik
    vec1 = np.array([p2[0] - p1[0], p2[1] - p1[1]])
    vec2 = np.array([p3[0] - p1[0], p3[1] - p1[1]])

    # Hitung dot product dan magnitudo kedua vektor
    dot_product = np.dot(vec1, vec2)
    magnitude1 = np.linalg.norm(vec1)
    magnitude2 = np.linalg.norm(vec2)

    # Hitung sudut dalam radian dan konversikan ke derajat
    cos_angle = dot_product / (magnitude1 * magnitude2)
    angle = np.arccos(cos_angle) * (180.0 / np.pi)
    return angle

# Fungsi untuk menghitung fitur landmark
def landmarkfeature(landmarks_img, labels):
    landmark_features = []
    
    for landmarks, label in zip(landmarks_img, labels):
        rahang_kiri = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[9, 0], landmarks[9, 1])) + line_length((landmarks[9, 0], landmarks[9, 1]), (landmarks[10, 0], landmarks[10, 1])) + line_length((landmarks[10, 0], landmarks[10, 1]), (landmarks[11, 0], landmarks[11, 1])) + line_length((landmarks[11, 0], landmarks[11, 1]), (landmarks[12, 0], landmarks[12, 1])) + line_length((landmarks[12, 0], landmarks[12, 1]), (landmarks[13, 0], landmarks[13, 1])) + line_length((landmarks[13, 0], landmarks[13, 1]), (landmarks[14, 0], landmarks[14, 1]))
        rahang_kanan = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[7, 0], landmarks[7, 1])) + line_length((landmarks[7, 0], landmarks[7, 1]), (landmarks[6, 0], landmarks[6, 1])) + line_length((landmarks[6, 0], landmarks[6, 1]), (landmarks[5, 0], landmarks[5, 1])) + line_length((landmarks[5, 0], landmarks[5, 1]), (landmarks[4, 0], landmarks[4, 1])) + line_length((landmarks[4, 0], landmarks[4, 1]), (landmarks[3, 0], landmarks[3, 1])) + line_length((landmarks[3, 0], landmarks[3, 1]), (landmarks[2, 0], landmarks[2, 1]))
        rahang = (rahang_kanan + rahang_kiri)/2
        atas_kiri = line_length((landmarks[16, 0], landmarks[16, 1]), (landmarks[78, 0], landmarks[78, 1])) + line_length((landmarks[78, 0], landmarks[78, 1]), (landmarks[74, 0], landmarks[74, 1])) + line_length((landmarks[74, 0], landmarks[74, 1]), (landmarks[79, 0], landmarks[79, 1])) + line_length((landmarks[79, 0], landmarks[79, 1]), (landmarks[73, 0], landmarks[73, 1])) + line_length((landmarks[73, 0], landmarks[73, 1]), (landmarks[72, 0], landmarks[72, 1]))

        atas_kanan = line_length((landmarks[0, 0], landmarks[0, 1]), (landmarks[77, 0], landmarks[77, 1])) + line_length((landmarks[77, 0], landmarks[77, 1]), (landmarks[75, 0], landmarks[75, 1])) + line_length((landmarks[75, 0], landmarks[75, 1]), (landmarks[76, 0], landmarks[76, 1])) + line_length((landmarks[76, 0], landmarks[76, 1]), (landmarks[68, 0], landmarks[68, 1])) + line_length((landmarks[68, 0], landmarks[68, 1]), (landmarks[69, 0], landmarks[69, 1]))
        atas = (atas_kanan + atas_kiri)/2
        # Hitung panjang 
        D0_length = line_length((landmarks[1, 0], landmarks[1, 1]), (landmarks[15, 0], landmarks[15, 1]))
        D1_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[71, 0], landmarks[71, 1]))
        D2_length = line_length((landmarks[75, 0], landmarks[75, 1]), (landmarks[79, 0], landmarks[79, 1]))
        D3_length = line_length((landmarks[2, 0], landmarks[2, 1]), (landmarks[14, 0], landmarks[14, 1]))
        D4_length = line_length((landmarks[3, 0], landmarks[3, 1]), (landmarks[13, 0], landmarks[13, 1]))
        D5_length = line_length((landmarks[4, 0], landmarks[4, 1]), (landmarks[12, 0], landmarks[12, 1]))
        D6_length = line_length((landmarks[5, 0], landmarks[5, 1]), (landmarks[11, 0], landmarks[11, 1]))
        D7_length = line_length((landmarks[6, 0], landmarks[6, 1]), (landmarks[10, 0], landmarks[10, 1]))
        D8_length = line_length((landmarks[7, 0], landmarks[7, 1]), (landmarks[9, 0], landmarks[9, 1]))

        D9_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[9, 0], landmarks[9, 1]))
        D91_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[7, 0], landmarks[7, 1]))
        
        D10_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[10, 0], landmarks[10, 1]))
        D101_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[6, 0], landmarks[6, 1]))
        
        D11_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[11, 0], landmarks[11, 1]))
        D111_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[5, 0], landmarks[5, 1]))
        
        D12_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[12, 0], landmarks[12, 1]))
        D121_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[4, 0], landmarks[4, 1]))
        
        D13_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[13, 0], landmarks[13, 1]))
        D131_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[3, 0], landmarks[3, 1]))
        
        D14_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[14, 0], landmarks[14, 1]))
        D141_length = line_length((landmarks[8, 0], landmarks[8, 1]), (landmarks[2, 0], landmarks[2, 1]))
        

        Sum_D = D1_length + D2_length + D3_length + D4_length + D5_length + D6_length + D7_length + D8_length + (D9_length + D91_length)/2 
        + (D10_length + D101_length)/2 + (D11_length + D111_length)/2 + (D12_length + D121_length)/2 + (D13_length + D131_length)/2 + (D14_length + D141_length)/2

        D0 = D0_length / Sum_D
        D1 = D1_length / Sum_D
        D2 = D2_length / Sum_D
        D3 = D3_length / Sum_D
        D4 = D4_length / Sum_D
        D5 = D5_length / Sum_D
        D6 = D6_length / Sum_D
        D7 = D7_length / Sum_D
        D8 = D8_length / Sum_D
        D9 = ((D9_length / Sum_D) + (D91_length / Sum_D)) / 2
        D10 = ((D10_length / Sum_D) + (D101_length / Sum_D)) / 2
        D11 = ((D11_length / Sum_D) + (D111_length / Sum_D)) / 2
        D12 = ((D12_length / Sum_D) + (D121_length / Sum_D)) / 2
        D13 = ((D13_length / Sum_D) + (D131_length / Sum_D)) / 2
        D14 = ((D14_length / Sum_D) + (D141_length / Sum_D)) / 2

        # Hitung rasio
        R1 = D0 / D3
        R2 = D0 / D4
        R3 = D0 / D5
        R4 = D0 / D6
        R5 = D0 / D7
        R6 = D0 / D8
        R7 = D6 / D1
        R8 = D5 / D2
        R9 = D4 / D5
        R10 = D7 / D6

        # Hitung sudut
        angle1a = hitungsudut((landmarks[8, 0], landmarks[8, 1]), (landmarks[71, 0], landmarks[71, 1]), (landmarks[10, 0], landmarks[10, 1]))
        angle1b = hitungsudut((landmarks[8, 0], landmarks[8, 1]), (landmarks[71, 0], landmarks[71, 1]), (landmarks[6, 0], landmarks[6, 1]))
        angle1 = (angle1a + angle1b)/2
        angle2a = hitungsudut((landmarks[8, 0], landmarks[8, 1]), (landmarks[71, 0], landmarks[71, 1]), (landmarks[12, 0], landmarks[12, 1]))
        angle2b = hitungsudut((landmarks[8, 0], landmarks[8, 1]), (landmarks[71, 0], landmarks[71, 1]), (landmarks[4, 0], landmarks[4, 1]))
        angle2 = (angle2a+angle2b)/2
        angle3a = hitungsudut((landmarks[14, 0], landmarks[14, 1]), (landmarks[2, 0], landmarks[2, 1]), (landmarks[12, 0], landmarks[12, 1]))
        angle3b = hitungsudut((landmarks[2, 0], landmarks[2, 1]), (landmarks[14, 0], landmarks[14, 1]), (landmarks[4, 0], landmarks[4, 1]))
        angle3 = (angle3a+angle3b)/2
        angle4a = hitungsudut((landmarks[9, 0], landmarks[9, 1]), (landmarks[8, 0], landmarks[8, 1]), (landmarks[10, 0], landmarks[10, 1]))
        angle4b = hitungsudut((landmarks[7, 0], landmarks[7, 1]), (landmarks[8, 0], landmarks[8, 1]), (landmarks[6, 0], landmarks[6, 1]))
        angle4 = (angle4a+angle4b)/2
        angle5a = hitungsudut((landmarks[11, 0], landmarks[11, 1]), (landmarks[10, 0], landmarks[10, 1]), (landmarks[12, 0], landmarks[12, 1]))
        angle5b = hitungsudut((landmarks[5, 0], landmarks[5, 1]), (landmarks[6, 0], landmarks[6, 1]), (landmarks[4, 0], landmarks[4, 1]))
        angle5 = (angle5a+angle5b)/2
        angle6a = hitungsudut((landmarks[13, 0], landmarks[13, 1]), (landmarks[12, 0], landmarks[12, 1]), (landmarks[14, 0], landmarks[14, 1]))
        angle6b = hitungsudut((landmarks[3, 0], landmarks[3, 1]), (landmarks[4, 0], landmarks[4, 1]), (landmarks[2, 0], landmarks[2, 1]))
        angle6 = (angle6a+angle6b)/2
        angle7 = hitungsudut((landmarks[8, 0], landmarks[8, 1]), (landmarks[7, 0], landmarks[7, 1]), (landmarks[9, 0], landmarks[9, 1]))

        landmark_features.append({
            'D1': D1,
            'D2': D2,
            'D3': D3,
            'D4': D4,
            'D5': D5,
            'D6': D6,
            'D7': D7,
            'D8': D8,
            'D9': D9,
            'D10': D10,
            'D11': D11,
            'D12': D12,
            'D13': D13,
            'D14': D14,
            'R1': R1,
            'R2': R2,
            'R3': R3,
            'R4': R4,
            'R5': R5,
            'R6': R6,
            'R7': R7,
            'R8': R8,
            'R9': R9,
            'R10': R10,
            'angle1': angle1,
            'angle2': angle2,
            'angle3': angle3,
            'angle4': angle4,
            'angle5': angle5,
            'angle6': angle6,
            'angle7': angle7,
            'rahang' : rahang,
            'atas' : atas,
            'label': label,
        })
    
    return landmark_features


# Menghitung fitur landmark untuk train dan test
train_landmarkfeature_img_list = landmarkfeature(train_landmark_images, train_std_label_list)
test_landmarkfeature_img_list = landmarkfeature(test_landmark_images, test_std_label_list)


C:\Users\ahmad\AppData\Local\Temp\ipykernel_9768\2495739644.py:17: RuntimeWarning: invalid value encountered in arccos
  angle = np.arccos(cos_angle) * (180.0 / np.pi)


In [10]:
# Ubah list dictionary ke dalam DataFrame untuk train dan test data
train_df = pd.DataFrame(train_landmarkfeature_img_list)
test_df = pd.DataFrame(test_landmarkfeature_img_list)

# Ekspor DataFrame ke CSV
train_df.to_csv('train_landmark_features.csv', index=False)
test_df.to_csv('test_landmark_features.csv', index=False)